# Day 5 — Random Forest Fights

This notebook is designed as a teaching notebook: we move from biological intuition to features, from labels to model training, and from model outputs to behavioral annotation in a new recording.

## Learning Goals

By the end, students should be able to:
- Explain why some features (e.g., inter-animal distance) can be more predictive of fights than others.
- Describe how temporal features are constructed from raw trajectories.
- Train and evaluate a Random Forest classifier on manually labeled events.
- Interpret model behavior using feature importance and threshold trade-offs.
- Understand how different annotators can change labels and model outcomes (what-if module).

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import joblib
import cv2

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score,
    precision_score, recall_score, f1_score
)

import QuantBio_functions as functions
import importlib
importlib.reload(functions)

%matplotlib inline

## 0) Data Setup (Noto-friendly)

This step ensures required files are present in `../quantBehData`. Missing files are downloaded automatically on first run; existing files are reused.

In [ ]:
RECORD_ID = "18835630"

data_bootstrap = functions.ensure_quant_beh_data_from_zenodo(record_id=RECORD_ID)

print("Downloaded files:", len(data_bootstrap["downloaded"]))
if data_bootstrap["missing_required"]:
    print("Still missing required files:")
    for name in data_bootstrap["missing_required"]:
        print("  -", name)

In [ ]:
# ----------------------
# Global settings (student-facing)
# ----------------------
DATA_ROOT = functions.get_quant_beh_data_dir()

candidate_manual_dirs = [
    os.path.abspath(os.path.join(os.getcwd(), "manualAnnotations_dense")),
    os.path.abspath(os.path.join(os.getcwd(), "trackingCourse", "manualAnnotations_dense")),
    os.path.abspath(os.path.join(os.getcwd(), "..", "trackingCourse", "manualAnnotations_dense")),
]
MANUAL_ANNOTATIONS_DIR = next((p for p in candidate_manual_dirs if os.path.isdir(p)), candidate_manual_dirs[0])

GT_IDS = ["gt1", "gt2", "gt3"]
NEW_DATASET_ID = "new_dataset"

FPS = 60
WINDOW_SIZE = 40

BASE_FEATURES = [
    "inter_animal_distance",
    "speed1", "speed2",
    "acc1", "acc2",
    "heading1", "heading2"
]
TEMPORAL_STATS = ["mean", "std", "max", "min", "delta"]

RF_PARAMS = {
    "n_estimators": 400,
    "class_weight": "balanced",
    "random_state": 42,
    "n_jobs": -1,
}

THRESHOLD = 0.4

# Annotation/video settings
ANNOTATION_WINDOW_MINUTES = 5
TRAIL_LENGTH = 10

print("DATA_ROOT:", DATA_ROOT)
print("MANUAL_ANNOTATIONS_DIR (dense):", MANUAL_ANNOTATIONS_DIR)
print("BASE_FEATURES:", BASE_FEATURES)
print("TEMPORAL_STATS per feature:", TEMPORAL_STATS)

## 1) Data Audit and Ground-Truth Inputs

Before training anything, we check whether expected files exist and inspect class balance in the teacher labels.

In [ ]:
required_files = []
for gt in GT_IDS:
    required_files.extend([
        os.path.join(DATA_ROOT, f"{gt}.mp4"),
        os.path.join(DATA_ROOT, f"{gt}_trajectories.csv"),
        os.path.join(DATA_ROOT, f"{gt}_manual_labeled_fights.csv"),
    ])

missing = [p for p in required_files if not os.path.exists(p)]
print("Total expected GT files:", len(required_files))
if missing:
    print("Missing files:")
    for p in missing:
        print("  -", p)
else:
    print("All expected GT files found.")

In [ ]:
teacher_labels_by_gt = {}
teacher_traj_by_gt = {}

for gt in GT_IDS:
    label_path = os.path.join(DATA_ROOT, f"{gt}_manual_labeled_fights.csv")
    traj_path = os.path.join(DATA_ROOT, f"{gt}_trajectories.csv")
    teacher_labels_by_gt[gt] = pd.read_csv(label_path)
    teacher_traj_by_gt[gt] = pd.read_csv(traj_path)

balance_rows = []
for gt, df in teacher_labels_by_gt.items():
    n_total = len(df)
    n_fight = int(df["label"].sum())
    balance_rows.append({
        "gt_id": gt,
        "n_frames": n_total,
        "n_fight_frames": n_fight,
        "fight_fraction": n_fight / n_total if n_total else np.nan
    })

balance_df = pd.DataFrame(balance_rows)
display(balance_df)

plt.figure(figsize=(6, 3))
sns.barplot(data=balance_df, x="gt_id", y="fight_fraction")
plt.title("Teacher label balance by recording")
plt.ylabel("Fraction of fight frames")
plt.xlabel("GT recording")
plt.tight_layout()
plt.show()

## 2) Feature Intuition Before ML

We inspect how basic features (distance, speed) relate to labeled fight periods in one GT recording.

In [ ]:
INSPECT_GT = "gt2"
FRAME_START = 0
FRAME_END = 4000

traj_df = teacher_traj_by_gt[INSPECT_GT].copy()
lab_df = teacher_labels_by_gt[INSPECT_GT].copy()

basic_df = functions.compute_basic_features(traj_df, fps=FPS)
merged_df = pd.merge(basic_df, lab_df[["frame", "label"]], on="frame", how="left")
merged_df["label"] = merged_df["label"].fillna(0)

view_df = merged_df[(merged_df["frame"] >= FRAME_START) & (merged_df["frame"] <= FRAME_END)].copy()

fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
axes[0].plot(view_df["frame"], view_df["inter_animal_distance"], lw=1)
axes[0].set_ylabel("Distance")
axes[0].set_title(f"Feature intuition on {INSPECT_GT}: distance, speed, and labels")
dist_vals = view_df["inter_animal_distance"].to_numpy()
dist_vals = dist_vals[np.isfinite(dist_vals)]
if dist_vals.size > 0:
    dist_ymax = np.percentile(dist_vals, 99) * 1.10
    if dist_ymax > 0:
        axes[0].set_ylim(0, dist_ymax)

axes[1].plot(view_df["frame"], view_df["speed1"], lw=1, label="speed1")
axes[1].plot(view_df["frame"], view_df["speed2"], lw=1, label="speed2")
axes[1].set_ylabel("Speed")
speed_vals = np.concatenate([view_df["speed1"].to_numpy(), view_df["speed2"].to_numpy()])
speed_vals = speed_vals[np.isfinite(speed_vals)]
if speed_vals.size > 0:
    speed_ymax = np.percentile(speed_vals, 99) * 1.10
    if speed_ymax > 0:
        axes[1].set_ylim(0, speed_ymax)
axes[1].legend(loc="upper right")

# Bottom panel: teacher labels + three student label overlays (offset by 10%, 20%, 30%).
axes[2].plot(view_df["frame"], view_df["label"], lw=1, color="red", label="teacher (+0.0)")

student_files_for_gt = functions.list_student_dense_annotation_files(
    MANUAL_ANNOTATIONS_DIR,
    gt_ids=[INSPECT_GT]
).get(INSPECT_GT, [])

for idx, (student_name, student_path) in enumerate(student_files_for_gt[:3], start=1):
    student_df = pd.read_csv(student_path)
    student_df = student_df[["frame", "label"]].copy()
    student_df["label"] = student_df["label"].fillna(0).astype(int)

    student_view = pd.merge(
        view_df[["frame"]],
        student_df,
        on="frame",
        how="left"
    )
    student_view["label"] = student_view["label"].fillna(0)

    y_offset = 0.1 * idx
    axes[2].plot(
        student_view["frame"],
        student_view["label"] + y_offset,
        lw=1,
        alpha=0.9,
        label=f"{student_name} (+{y_offset:.1f})"
    )

axes[2].set_ylabel("Fight label")
axes[2].set_xlabel("Frame")
axes[2].set_ylim(-0.05, 1.35)
axes[2].legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.show()

## 3) Temporal Feature Construction

Each base feature is summarized over a rolling window (`WINDOW_SIZE`) using: mean, std, max, min, and delta.

In [ ]:
print("Base features used:", BASE_FEATURES)
print("Temporal summary stats:", TEMPORAL_STATS)
print("Window size (frames):", WINDOW_SIZE)
print("Approx window duration (seconds):", WINDOW_SIZE / FPS)

def build_temporal_labeled_df(traj_df, label_df, fps=FPS, window_size=WINDOW_SIZE):
    basic = functions.compute_basic_features(traj_df, fps=fps)
    merged = pd.merge(basic, label_df[["frame", "label"]], on="frame", how="inner")
    temporal = functions.generate_temporal_features(merged, BASE_FEATURES, window_size=window_size)
    temporal = pd.merge(temporal, merged[["frame", "label"]], on="frame", how="left")
    return temporal

## 4) Teacher-Labeled Training Table

Build temporal features for each GT recording using teacher labels.

In [ ]:
teacher_temporal_by_gt = {}
for gt in GT_IDS:
    teacher_temporal_by_gt[gt] = build_temporal_labeled_df(
        teacher_traj_by_gt[gt],
        teacher_labels_by_gt[gt],
        fps=FPS,
        window_size=WINDOW_SIZE
    )
    print(gt, teacher_temporal_by_gt[gt].shape)

teacher_feature_columns = [c for c in teacher_temporal_by_gt[GT_IDS[0]].columns if c not in ["frame", "label"]]
print("Number of temporal features:", len(teacher_feature_columns))

## 5) Student Annotations (Selectable, No Extra Dependencies)

Students choose one annotator per GT recording by editing a simple dictionary below.

In [ ]:
student_files_by_gt = functions.list_student_dense_annotation_files(
    MANUAL_ANNOTATIONS_DIR,
    gt_ids=GT_IDS
)

functions.print_student_dense_annotation_options(
    student_files_by_gt,
    gt_ids=GT_IDS
)

In [ ]:
# Edit these indices to select one student annotator per GT recording
STUDENT_SELECTION_INDEX = {
    "gt1": 0,
    "gt2": 0,
    "gt3": 0,
}

selected_students = functions.resolve_student_selection(
    student_files_by_gt,
    STUDENT_SELECTION_INDEX,
    gt_ids=GT_IDS
)

print("Selected student annotators:")
for gt, (name, path) in selected_students.items():
    print(f"  {gt}: {name}")

In [ ]:
student_labels_by_gt = functions.load_selected_student_labels_by_gt(
    selected_students,
    teacher_labels_by_gt,
    gt_ids=GT_IDS,
    verbose=True
)

student_temporal_by_gt = {}
for gt in GT_IDS:
    student_temporal_by_gt[gt] = build_temporal_labeled_df(
        teacher_traj_by_gt[gt],
        student_labels_by_gt[gt],
        fps=FPS,
        window_size=WINDOW_SIZE
    )
    print(gt, student_temporal_by_gt[gt].shape)

## 6) Teacher vs Selected Student Label Agreement

This quantifies how much the selected student labels align with the teacher labels before model training.

In [ ]:
agreement_rows = []
for gt in GT_IDS:
    t = teacher_labels_by_gt[gt].copy()
    s = student_labels_by_gt[gt].copy()
    m = pd.merge(t, s, on="frame", suffixes=("_teacher", "_student"), how="inner")

    y_t = m["label_teacher"].to_numpy()
    y_s = m["label_student"].to_numpy()

    agreement_rows.append({
        "gt_id": gt,
        "accuracy": accuracy_score(y_t, y_s),
        "precision_student_vs_teacher": precision_score(y_t, y_s, zero_division=0),
        "recall_student_vs_teacher": recall_score(y_t, y_s, zero_division=0),
        "f1_student_vs_teacher": f1_score(y_t, y_s, zero_division=0),
    })

agreement_df = pd.DataFrame(agreement_rows)
display(agreement_df)

## 7) Train/Test Protocol (Split by Recording)

To avoid leakage from nearby frames, we hold out one full recording for testing.

In [ ]:
HOLDOUT_GT = "gt1"  # change to gt2 or gt3 for what-if
print("Holdout recording:", HOLDOUT_GT)

def train_eval_by_recording(temporal_by_gt, holdout_gt, rf_params, threshold):
    train_parts = []
    test_df = temporal_by_gt[holdout_gt].copy()

    for gt, df in temporal_by_gt.items():
        if gt != holdout_gt:
            train_parts.append(df)

    train_df = pd.concat(train_parts, ignore_index=True)

    feature_cols = [c for c in train_df.columns if c not in ["frame", "label"]]

    X_train = train_df[feature_cols]
    y_train = train_df["label"]
    X_test = test_df[feature_cols]
    y_test = test_df["label"]

    clf = RandomForestClassifier(**rf_params)
    clf.fit(X_train, y_train)

    y_prob = clf.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "fpr": fp / (fp + tn) if (fp + tn) else np.nan,
        "fnr": fn / (fn + tp) if (fn + tp) else np.nan,
        "n_train": len(train_df),
        "n_test": len(test_df),
    }

    test_out = test_df.copy()
    test_out["fight_probability"] = y_prob
    test_out["predicted_label"] = y_pred

    return clf, feature_cols, metrics, test_out

In [ ]:
teacher_clf, teacher_features, teacher_metrics, teacher_test_out = train_eval_by_recording(
    teacher_temporal_by_gt, HOLDOUT_GT, RF_PARAMS, THRESHOLD
)

student_clf, student_features, student_metrics, student_test_out = train_eval_by_recording(
    student_temporal_by_gt, HOLDOUT_GT, RF_PARAMS, THRESHOLD
)

compare_df = pd.DataFrame([
    {"label_source": "teacher", **teacher_metrics},
    {"label_source": "selected_student", **student_metrics},
])
display(compare_df)

## 8) What Features Did the Model Use Most?

Random Forest feature importances provide one view of which temporal features helped classification.

In [ ]:
def top_importance_df(clf, feature_cols, top_n=15):
    imp = pd.DataFrame({
        "feature": feature_cols,
        "importance": clf.feature_importances_
    }).sort_values("importance", ascending=False)
    return imp.head(top_n)

teacher_top = top_importance_df(teacher_clf, teacher_features, top_n=15)
student_top = top_importance_df(student_clf, student_features, top_n=15)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=teacher_top, x="importance", y="feature", ax=axes[0])
axes[0].set_title("Teacher model: top importances")
sns.barplot(data=student_top, x="importance", y="feature", ax=axes[1])
axes[1].set_title("Selected-student model: top importances")
plt.tight_layout()
plt.show()

## 9) Threshold What-If

Changing threshold changes false positives vs false negatives.

In [ ]:
threshold_grid = np.linspace(0.1, 0.9, 17)
rows = []

y_true = teacher_test_out["label"].to_numpy()
y_prob = teacher_test_out["fight_probability"].to_numpy()

for th in threshold_grid:
    y_hat = (y_prob >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_hat, labels=[0, 1]).ravel()
    rows.append({
        "threshold": th,
        "precision": precision_score(y_true, y_hat, zero_division=0),
        "recall": recall_score(y_true, y_hat, zero_division=0),
        "f1": f1_score(y_true, y_hat, zero_division=0),
        "fpr": fp / (fp + tn) if (fp + tn) else np.nan,
        "fnr": fn / (fn + tp) if (fn + tp) else np.nan,
    })

th_df = pd.DataFrame(rows)
display(th_df.head())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(th_df["threshold"], th_df["precision"], label="precision")
axes[0].plot(th_df["threshold"], th_df["recall"], label="recall")
axes[0].plot(th_df["threshold"], th_df["f1"], label="f1")
axes[0].axvline(THRESHOLD, color="black", linestyle="--", label="chosen")
axes[0].set_title("Metrics vs threshold")
axes[0].set_xlabel("threshold")
axes[0].legend()

axes[1].plot(th_df["threshold"], th_df["fpr"], label="FPR")
axes[1].plot(th_df["threshold"], th_df["fnr"], label="FNR")
axes[1].axvline(THRESHOLD, color="black", linestyle="--", label="chosen")
axes[1].set_title("Error trade-off vs threshold")
axes[1].set_xlabel("threshold")
axes[1].legend()

plt.tight_layout()
plt.show()

## 10) Apply to a New Recording

We generate temporal features for a new dataset, run prediction, and save outputs in `quantBehData`.

In [ ]:
new_temporal_csv = functions.process_dataset_id_from_flat_root(
    data_root=DATA_ROOT,
    dataset_id=NEW_DATASET_ID,
    fps=FPS,
    window_size=WINDOW_SIZE,
    require_labels=False
)

new_df = pd.read_csv(new_temporal_csv)

# Align to teacher model features
missing_cols = [c for c in teacher_features if c not in new_df.columns]
if missing_cols:
    raise ValueError(f"New temporal features missing columns: {missing_cols}")

X_new = new_df[teacher_features]
new_prob = teacher_clf.predict_proba(X_new)[:, 1]
new_pred = (new_prob >= THRESHOLD).astype(int)

new_df["fight_probability"] = new_prob
new_df["predicted_label"] = new_pred

teacher_pred_path = os.path.join(DATA_ROOT, f"{NEW_DATASET_ID}_predictions_teacher.csv")
new_df.to_csv(teacher_pred_path, index=False)
print("Saved teacher-model predictions:", teacher_pred_path)

In [ ]:
pred_df = pd.read_csv(teacher_pred_path)

fight_prob = pred_df["fight_probability"].to_numpy()
prob_2d = fight_prob[np.newaxis, :]

cmap = LinearSegmentedColormap.from_list("lightblue_red", ["#e6f2ff", "red"])

fig, ax = plt.subplots(figsize=(15, 3))
im = ax.imshow(prob_2d, aspect="auto", cmap=cmap, vmin=0, vmax=1, interpolation="nearest")
ax.set_yticks([])
ax.set_xlabel("Frame")
cbar = plt.colorbar(im, ax=ax, orientation="vertical", pad=0.02)
cbar.set_label("Fight probability", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
annotated_video_path = os.path.join(DATA_ROOT, f"{NEW_DATASET_ID}_annotated_teacher.mp4")

functions.annotate_most_fight_chunk(
    video_path=os.path.join(DATA_ROOT, f"{NEW_DATASET_ID}.mp4"),
    trajectory_csv=os.path.join(DATA_ROOT, f"{NEW_DATASET_ID}_trajectories.csv"),
    predictions_csv=teacher_pred_path,
    output_path=annotated_video_path,
    window_minutes=ANNOTATION_WINDOW_MINUTES,
    threshold=THRESHOLD,
    trail_length=TRAIL_LENGTH
)

print("Saved annotated video:", annotated_video_path)

## 11) End Module — All Annotators Consensus (What-If)

Now we use *all* student annotators for each GT recording, build a consensus label, and compare model outcomes.

In [ ]:
CONSENSUS_THRESHOLD = 0.5  # fraction of annotators that must mark a frame as fight

consensus_labels_by_gt = {}
for gt in GT_IDS:
    n_total = len(teacher_labels_by_gt[gt])
    consensus_labels_by_gt[gt] = functions.build_consensus_labels_for_gt(
        gt, student_files_by_gt, n_total, CONSENSUS_THRESHOLD
    )

    print(gt, "annotators:", len(student_files_by_gt[gt]),
          "fight fraction (consensus):", consensus_labels_by_gt[gt]["label"].mean())

In [ ]:
consensus_temporal_by_gt = {}
for gt in GT_IDS:
    consensus_temporal_by_gt[gt] = build_temporal_labeled_df(
        teacher_traj_by_gt[gt],
        consensus_labels_by_gt[gt][["frame", "label"]],
        fps=FPS,
        window_size=WINDOW_SIZE
    )

consensus_clf, consensus_features, consensus_metrics, consensus_test_out = train_eval_by_recording(
    consensus_temporal_by_gt, HOLDOUT_GT, RF_PARAMS, THRESHOLD
)

final_compare_df = pd.DataFrame([
    {"label_source": "teacher", **teacher_metrics},
    {"label_source": "selected_student", **student_metrics},
    {"label_source": "student_consensus", **consensus_metrics},
])
display(final_compare_df)

## 12) Reflection Questions

- Which temporal features consistently appear in top importances?
- How sensitive are your conclusions to threshold choice?
- How much do model metrics shift when labels come from different annotators?
- If two models perform similarly, which one would you trust more and why?